# 4.6 Obstacle Investigation

Per-tile 2-D map of the labeled point cloud with all extracted clusters overlaid.  
**Click any cluster dot** to see its top-view and side-view in the right panels.

Use the **Tile** dropdown to switch between tiles.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import numpy as np
import pandas as pd
import laspy
import ipywidgets as widgets
from IPython.display import display
%matplotlib widget
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from config import CLUSTERS_DIR, LABELED_DIR

In [ ]:
inv = pd.read_csv(CLUSTERS_DIR / 'inventory.csv')
TILECODES = sorted(inv['tilecode'].unique().tolist())
print(f'Clusters: {len(inv)}   Tiles: {TILECODES}')

In [ ]:
# ── label colours ─────────────────────────────────────────────────────────────
LABEL_NAMES = {
    0:  'Unknown',
    1:  'Road',
    9:  'Ground',
    10: 'Building',
    30: 'Tree',
    40: 'Car',
    60: 'Street Light',
    83: 'Large Container',
}

# Background label → RGB for the 2-D grid
BG_RGB = {
    -1: (0.07, 0.07, 0.07),   # empty cell
     0: (0.25, 0.25, 0.25),   # unknown
     1: (0.75, 0.20, 0.20),   # road
     9: (0.55, 0.55, 0.55),   # other ground
    10: (0.20, 0.35, 0.70),   # building
    30: (0.20, 0.65, 0.20),   # tree
    40: (1.00, 0.50, 0.10),   # car
    60: (1.00, 0.95, 0.20),   # street light
    79: (0.80, 0.40, 0.00),   # cable
    83: (0.70, 0.20, 0.70),   # large container
    90: (0.90, 0.60, 0.10),   # armatuur
}

# Priority: which label wins when multiple fall in the same cell
# Higher number = higher priority (drawn last, wins)
LABEL_PRIORITY = {10: 8, 1: 7, 30: 6, 40: 5, 60: 4, 83: 4, 79: 3, 90: 3, 9: 2, 0: 1}

# Cluster centroid colours
DOT_COLORS = {
    0:  '#aaaaaa',
    30: '#44ee44',
    40: '#ff8800',
    60: '#44aaff',
    83: '#dd44dd',
}
DOT_DEFAULT = '#ffffff'


def hag_colors(npz):
    xyz = npz['xyz_centered']
    hag = npz.get('height_ag', None)
    if hag is None or np.all(np.isnan(hag)):
        hag = xyz[:, 2] - xyz[:, 2].min()
    hag = np.nan_to_num(hag, nan=0.0)
    return plt.cm.plasma(np.clip(hag, 0, 6) / 6.0)


def style_ax(ax):
    ax.set_facecolor('#111111')
    for sp in ax.spines.values():
        sp.set_color('#333333')
    ax.tick_params(colors='#777777', labelsize=7)

In [ ]:
# ── background grid builder ───────────────────────────────────────────────────
GRID_RES = 0.25  # metres per pixel
_tile_cache = {}


def _load_tile_raw(tilecode):
    """Load xyz + labels from bgt_labeled LAZ; cache result."""
    if tilecode in _tile_cache:
        return _tile_cache[tilecode]
    laz_path = LABELED_DIR / f'bgt_labeled_{tilecode}.laz'
    print(f'  Loading {laz_path.name} …', end=' ', flush=True)
    pc = laspy.read(laz_path)
    xyz = np.column_stack([
        np.asarray(pc.x, dtype=np.float32),
        np.asarray(pc.y, dtype=np.float32),
    ])
    has_lbl = 'label' in pc.point_format.extra_dimension_names
    lbl = np.asarray(pc.label, dtype=np.int32) if has_lbl else np.zeros(len(xyz), dtype=np.int32)
    print(f'{len(xyz):,} pts')
    _tile_cache[tilecode] = (xyz, lbl)
    return xyz, lbl


def make_bg_image(xy, labels, res=GRID_RES):
    """Bin points into a 2-D label grid and return an RGB image + extent."""
    x, y = xy[:, 0], xy[:, 1]
    x_min, y_min = float(x.min()), float(y.min())

    xi = np.floor((x - x_min) / res).astype(np.int32)
    yi = np.floor((y - y_min) / res).astype(np.int32)
    nx, ny = int(xi.max()) + 1, int(yi.max()) + 1

    # Sort by label priority so high-priority labels overwrite lower ones
    prio = np.vectorize(lambda l: LABEL_PRIORITY.get(int(l), 0))(labels)
    order = np.argsort(prio)  # ascending → highest priority written last

    grid = np.full((ny, nx), -1, dtype=np.int32)
    grid[yi[order], xi[order]] = labels[order]

    # Build RGB image
    default_rgb = BG_RGB.get(-1)
    rgb = np.full((ny, nx, 3), default_rgb, dtype=np.float32)
    for lbl_val, color in BG_RGB.items():
        mask = grid == lbl_val
        if mask.any():
            rgb[mask] = color

    extent = [x_min, x_min + nx * res, y_min, y_min + ny * res]
    return rgb, extent

In [ ]:
# ── figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(17, 7))
fig.patch.set_facecolor('#111111')

gs = gridspec.GridSpec(2, 2, figure=fig,
                       width_ratios=[1.8, 1],
                       hspace=0.45, wspace=0.35)
ax_map  = fig.add_subplot(gs[:, 0])
ax_top  = fig.add_subplot(gs[0, 1])
ax_side = fig.add_subplot(gs[1, 1])

for ax in (ax_map, ax_top, ax_side):
    style_ax(ax)

ax_top.set_title('top view  (click a cluster)', color='#555555', fontsize=9)
ax_side.set_title('side view', color='#555555', fontsize=9)

# persistent artists on ax_map
_bg_im       = ax_map.imshow(np.zeros((1, 1, 3)), origin='lower',
                              aspect='equal', interpolation='nearest')
_sel_marker, = ax_map.plot([], [], 'w*', markersize=16, zorder=6,
                           markeredgecolor='#ff3333', markeredgewidth=1.2)
_info_text   = ax_map.text(
    0.01, 0.01, 'click a cluster dot to inspect',
    transform=ax_map.transAxes, color='#999999', fontsize=8, va='bottom',
    bbox=dict(facecolor='#1a1a1a', edgecolor='none', alpha=0.8, pad=4),
)

# scatter artists per label (rebuilt on tile switch)
_dot_artists = []
_current_tile_inv = [None]  # mutable container for current-tile clusters


# ── tile renderer ─────────────────────────────────────────────────────────────
def render_tile(tilecode):
    global _dot_artists

    # background
    xy, lbl = _load_tile_raw(tilecode)
    rgb, extent = make_bg_image(xy, lbl)
    _bg_im.set_data(rgb)
    _bg_im.set_extent(extent)
    ax_map.set_xlim(extent[0], extent[1])
    ax_map.set_ylim(extent[2], extent[3])
    ax_map.set_title(f'{tilecode} — click a cluster', color='white', fontsize=10)
    ax_map.set_xlabel('X (m RD)', color='#777777', fontsize=8)
    ax_map.set_ylabel('Y (m RD)', color='#777777', fontsize=8)

    # remove old cluster dots
    for sc in _dot_artists:
        sc.remove()
    _dot_artists.clear()

    # draw cluster centroids for this tile
    tile_inv = inv[inv['tilecode'] == tilecode].reset_index(drop=True)
    _current_tile_inv[0] = tile_inv
    for lbl_code, grp in tile_inv.groupby('label'):
        color = DOT_COLORS.get(lbl_code, DOT_DEFAULT)
        name  = LABEL_NAMES.get(lbl_code, f'Label {lbl_code}')
        sc = ax_map.scatter(
            grp['centroid_x'], grp['centroid_y'],
            c=color, s=90, zorder=5, label=name,
            edgecolors='#111111', linewidths=0.6, alpha=0.95,
        )
        _dot_artists.append(sc)

    legend = ax_map.legend(
        facecolor='#1e1e1e', labelcolor='white',
        edgecolor='#444444', fontsize=8, loc='upper right',
    )

    # reset selection
    _sel_marker.set_data([], [])
    _info_text.set_text('click a cluster dot to inspect')
    fig.canvas.draw_idle()


# ── click handler ─────────────────────────────────────────────────────────────
def on_click(event):
    if event.inaxes is not ax_map:
        return
    if event.xdata is None or event.ydata is None:
        return

    tile_inv = _current_tile_inv[0]
    if tile_inv is None or len(tile_inv) == 0:
        return

    dist = np.hypot(tile_inv['centroid_x'] - event.xdata,
                    tile_inv['centroid_y'] - event.ydata)
    row = tile_inv.loc[dist.idxmin()]

    _sel_marker.set_data([row['centroid_x']], [row['centroid_y']])

    lbl  = int(row['label'])
    name = LABEL_NAMES.get(lbl, f'Label {lbl}')
    _info_text.set_text(
        f"#{row['cluster_idx']}  {name}  ·  "
        f"{int(row['n_raw_pts']):,} pts  ·  {float(row['area_m2']):.2f} m²  ·  "
        f"{row.get('label_source', '')}"
    )

    try:
        npz = np.load(row['npz_path'])
    except Exception as e:
        _info_text.set_text(f'Error: {e}')
        fig.canvas.draw_idle()
        return

    xyz  = npz['xyz_centered']
    cols = hag_colors(npz)
    detail_title = (
        f"#{row['cluster_idx']}  {name}  ·  "
        f"{int(row['n_raw_pts']):,} pts  ·  {float(row['area_m2']):.2f} m²"
    )

    ax_top.cla()
    style_ax(ax_top)
    ax_top.scatter(xyz[:, 0], xyz[:, 1], c=cols, s=1.5, linewidths=0)
    ax_top.set_aspect('equal')
    ax_top.set_title(detail_title, color='white', fontsize=7)
    ax_top.text(0.02, 0.97, 'top (XY)', transform=ax_top.transAxes,
                color='#888888', fontsize=6, va='top')

    ax_side.cla()
    style_ax(ax_side)
    ax_side.scatter(xyz[:, 0], xyz[:, 2], c=cols, s=1.5, linewidths=0)
    ax_side.set_aspect('equal')
    ax_side.text(0.02, 0.97, 'side (XZ)', transform=ax_side.transAxes,
                color='#888888', fontsize=6, va='top')

    fig.canvas.draw_idle()


fig.canvas.mpl_connect('button_press_event', on_click)


# ── tile dropdown ─────────────────────────────────────────────────────────────
tile_dd = widgets.Dropdown(
    options=TILECODES, value=TILECODES[0],
    description='Tile:', layout=widgets.Layout(width='320px'),
    style={'description_width': '40px'},
)

def on_tile_change(change):
    if change['name'] == 'value':
        render_tile(change['new'])

tile_dd.observe(on_tile_change)

display(tile_dd)
plt.show()

# load first tile immediately
render_tile(TILECODES[0])

### Colour legend

| Colour | Label |
|---|---|
| 🔴 Dark red | Road |
| 🔵 Blue | Building |
| ⚫ Dark grey | Ground / unknown |
| 🟢 Green | Tree |
| 🟠 Orange | Car |
| 🟡 Yellow | Street light |

Cluster dots use the same colours. The white ★ marks the currently selected cluster.